# Cavitation and EOS Sweep Driver

This notebook is a thin driver around the current V3 helpers. The cavitation sweep uses the existing get-or-create chain:

```text
cavitation evolution -> cavitation initial state -> thermalized state -> FCC lattice
```

Rows are intentionally lightweight unless they are scientifically interesting: a non-phase-separated thermalized source followed by a phase-separated cavitation final state.

In [2]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from md_Helpers import cavitation, eos_sweep, paths, seitz
from md_Helpers.cavitation_sweep import run_cavitation_size_sweep

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 120)

## Cavitation Inputs

In [3]:
# Edit these first.
cavitation_densities = [0.70, 0.71, 0.72]
cavitation_temperatures = [0.80]
cavitation_n_fcc_cells = [10, 15, 20, 25, 30]
starting_radii = [3, 4, 5]

source_nsteps = 1_000_000
evolve_nsteps = 100_000
source_seed = 1
evolve_seeds = [1]

trajectory_period = 1_000
log_period = 1_000

cavitation_summary_dir = paths.MASTER_CSVS_V3_ROOT / "cavitation_radius_sweeps"
cavitation_summary_path = cavitation_summary_dir / "cavitation_interesting_sweep_all_radii.csv"

cavitation_conditions = [
    {
        "density": rho,
        "temperature": kT,
        "label": f"rho={rho:.3f}, kT={kT:.3f}",
    }
    for kT in cavitation_temperatures
    for rho in cavitation_densities
]

cavitation_conditions

[{'density': 0.7, 'temperature': 0.8, 'label': 'rho=0.700, kT=0.800'},
 {'density': 0.71, 'temperature': 0.8, 'label': 'rho=0.710, kT=0.800'},
 {'density': 0.72, 'temperature': 0.8, 'label': 'rho=0.720, kT=0.800'}]

In [ ]:
radius_tables = []

for starting_radius in starting_radii:
    radius_summary_path = (
        cavitation_summary_dir
        / f"cavitation_interesting_sweep_radius_{starting_radius:.3f}.csv"
    )
    table = run_cavitation_size_sweep(
        n_fcc_cells_values=cavitation_n_fcc_cells,
        conditions=cavitation_conditions,
        source_nsteps=source_nsteps,
        evolve_nsteps=evolve_nsteps,
        radius=starting_radius,
        evolve_seeds=evolve_seeds,
        source_seed=source_seed,
        trajectory_period=trajectory_period,
        log_period=log_period,
        summary_path=radius_summary_path,
        summary_mode="interesting_only",
    )
    table["starting_radius"] = float(starting_radius)
    radius_tables.append(table)

cavitation_summary = pd.concat(radius_tables, ignore_index=True)
cavitation_summary_path.parent.mkdir(parents=True, exist_ok=True)
cavitation_summary.to_csv(cavitation_summary_path, index=False)

cavitation_summary

Thermalized state does not exist: running now.
No source thermalized state found for the specified values.
n_fcc_cells       = 10
target_rho        = 0.7
kT                = 0.8
source_nsteps     = 1000000
source_seed       = 1
source_phase_name = randomization

Missing paths:
/exp/e961/data/MDsims-data/pnichols/Thermalized_States_v3/FCC/n_cells_10/rho_0.700/kT_0.800/nsteps_1000000/seed_1/randomization.gsd
/exp/e961/data/MDsims-data/pnichols/Thermalized_States_v3/FCC/n_cells_10/rho_0.700/kT_0.800/nsteps_1000000/seed_1/randomization_log.hdf5
create_source_if_missing=True; starting thermalization now.
Created new FCC lattice
----------------------------
n_fcc_cells = 10
N = 4000
Target rho = 0.7
Actual rho = 0.7000000000000004
Density error = 4.440892098500626e-16
BoxLength = 17.87807070193135
FCC cell size = 1.787807070193135
Saved lattice to:
/exp/e961/data/MDsims-data/pnichols/Simple_Lattices_v3/FCC/n_cells_10/rho_0.700/lattice.gsd
Using GPU device
Final device: <hoomd.device.GPU obje

In [ ]:
interesting_cavitations = cavitation_summary[
    (cavitation_summary["thermalization_passed"].astype(bool))
    & (cavitation_summary["final_phase_separated"].astype(bool))
].copy()

status_counts = (
    cavitation_summary
    .groupby(["run_status", "outcome"], dropna=False)
    .size()
    .reset_index(name="count")
)

display(status_counts)
interesting_cavitations

## EOS Inputs

This section compares the liquid EOS quantities used as `P0` and `u0` inputs. It runs one fixed temperature across a density window for each selected FCC size.

In [ ]:
# Edit these for the EOS comparison.
eos_n_fcc_cells = [10, 15, 20]
eos_kT = 0.80
eos_density_start = 0.68
eos_density_stop = 0.74
eos_density_step = 0.005

eos_nsteps = source_nsteps
eos_seed = source_seed
eos_log_period = log_period
eos_n_last = 100

eos_output_dir = paths.MASTER_CSVS_V3_ROOT / "EOS_ncells_compare"

In [ ]:
eos_tables = []

for n_cells in eos_n_fcc_cells:
    output_path = eos_output_dir / f"eos_kT_{eos_kT:.3f}_ncells_{int(n_cells)}.csv"
    table = eos_sweep.run_eos_pressure_window_sweep(
        n_fcc_cells=n_cells,
        kT_start=eos_kT,
        kT_stop=eos_kT,
        kT_step=1.0,
        initial_rho=eos_density_start,
        rho_step=eos_density_step,
        rho_min_hard=eos_density_start,
        rho_max_hard=eos_density_stop,
        lower_stop=-1.0e9,
        upper_stop=1.0e9,
        nsteps=eos_nsteps,
        log_period=eos_log_period,
        seed=eos_seed,
        n_last=eos_n_last,
        output_path=output_path,
    )
    eos_tables.append(table)

eos_combined = pd.concat(eos_tables, ignore_index=True)
eos_combined

In [ ]:
eos_liquid = eos_sweep.liquid_eos_table(
    eos_combined,
    n_last=eos_n_last,
    n_fcc_cells=None,
)

pressure_col = f"pressure_mean_last{eos_n_last}"
pressure_std_col = f"pressure_std_last{eos_n_last}"
u0_col = f"PE_per_particle_mean_last{eos_n_last}"
u0_std_col = f"PE_per_particle_std_last{eos_n_last}"

eos_liquid[[
    "n_fcc_cells",
    "actual_rho",
    "kT",
    pressure_col,
    u0_col,
    "phase_separated",
    "log_path",
]]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharex=True)

for n_cells, group in eos_liquid.groupby("n_fcc_cells"):
    group = group.sort_values("actual_rho")
    axes[0].errorbar(
        group["actual_rho"],
        group[pressure_col],
        yerr=group.get(pressure_std_col),
        marker="o",
        capsize=3,
        label=f"n={int(n_cells)}",
    )
    axes[1].errorbar(
        group["actual_rho"],
        group[u0_col],
        yerr=group.get(u0_std_col),
        marker="o",
        capsize=3,
        label=f"n={int(n_cells)}",
    )

axes[0].axhline(0.0, color="black", linewidth=1, alpha=0.35)
axes[0].set_ylabel("P0: pressure")
axes[1].set_ylabel("u0: PE / particle")

for ax in axes:
    ax.set_xlabel("Actual density, N / V")
    ax.grid(alpha=0.3)
    ax.legend(title="FCC cells")

fig.suptitle(f"EOS comparison at kT = {eos_kT:.3f}")
fig.tight_layout()

plot_path = eos_output_dir / f"eos_compare_kT_{eos_kT:.3f}.png"
plot_path.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(plot_path, dpi=160)
plot_path

## Pressure-Selected Seitz Sweep

This repeats the pressure-window EOS selection for a range of `n_fcc_cells` at one temperature. Each non-phase-separated EOS density whose pressure falls inside the requested window is then used as a cavitation source for a small list of starting radii, and Seitz terms are computed from those completed cavitations.

In [2]:
# Edit these for pressure-selected Seitz runs.
seitz_n_fcc_cells = [10, 15, 20, 25, 30]
seitz_reference_eos_n_fcc_cells = 30
seitz_kT = 0.80

# Starting guess for the adaptive pressure sweep. The sweep moves up/down
# from here until it brackets the requested pressure range.
seitz_initial_rho = 0.71
seitz_rho_step = 0.005

pressure_lower = -0.03
pressure_upper = 0.18

seitz_starting_radii = [2.5, 3.0, 3.5, 4.0, 5.0]

seitz_source_nsteps = 1_000_000
seitz_evolve_nsteps = 100_000
seitz_source_seed = 1
seitz_evolve_seeds = [1]
seitz_log_period = 1_000
seitz_trajectory_period = 1_000
seitz_n_last = 100

seitz_output_dir = paths.MASTER_CSVS_V3_ROOT / "Pressure_Selected_Seitz"
seitz_eos_output_dir = seitz_output_dir / "EOS"
seitz_summary_path = seitz_output_dir / f"seitz_pressure_selected_kT_{seitz_kT:.3f}.csv"


In [ ]:
pressure_eos_tables = []

# Internal safety bounds for the adaptive pressure search. These are not the
# selected density range; the pressure window below decides which densities are used.
seitz_rho_min_hard = 0.50
seitz_rho_max_hard = 0.85

for n_cells in seitz_n_fcc_cells:
    output_path = seitz_eos_output_dir / f"eos_pressure_window_kT_{seitz_kT:.3f}_ncells_{int(n_cells)}.csv"
    table = eos_sweep.run_eos_pressure_window_sweep(
        n_fcc_cells=n_cells,
        kT_start=seitz_kT,
        kT_stop=seitz_kT,
        kT_step=1.0,
        initial_rho=seitz_initial_rho,
        rho_step=seitz_rho_step,
        rho_min_hard=seitz_rho_min_hard,
        rho_max_hard=seitz_rho_max_hard,
        lower_stop=pressure_lower,
        upper_stop=pressure_upper,
        nsteps=seitz_source_nsteps,
        log_period=seitz_log_period,
        seed=seitz_source_seed,
        n_last=seitz_n_last,
        output_path=output_path,
    )
    pressure_eos_tables.append(table)

pressure_eos = pd.concat(pressure_eos_tables, ignore_index=True)
pressure_col = f"pressure_mean_last{seitz_n_last}"

pressure_eos_liquid = eos_sweep.liquid_eos_table(
    pressure_eos,
    n_last=seitz_n_last,
    n_fcc_cells=None,
)

selected_eos_states = pressure_eos_liquid[
    pressure_eos_liquid[pressure_col].between(pressure_lower, pressure_upper)
].copy()

reference_eos_path = (
    seitz_eos_output_dir
    / f"reference_eos_kT_{seitz_kT:.3f}_ncells_{int(seitz_reference_eos_n_fcc_cells)}.csv"
)
reference_eos = eos_sweep.run_eos_pressure_window_sweep(
    n_fcc_cells=seitz_reference_eos_n_fcc_cells,
    kT_start=seitz_kT,
    kT_stop=seitz_kT,
    kT_step=1.0,
    initial_rho=seitz_initial_rho,
    rho_step=seitz_rho_step,
    rho_min_hard=seitz_rho_min_hard,
    rho_max_hard=seitz_rho_max_hard,
    lower_stop=pressure_lower,
    upper_stop=pressure_upper,
    nsteps=seitz_source_nsteps,
    log_period=seitz_log_period,
    seed=seitz_source_seed,
    n_last=seitz_n_last,
    output_path=reference_eos_path,
)
reference_eos_liquid = eos_sweep.liquid_eos_table(
    reference_eos,
    n_last=seitz_n_last,
    n_fcc_cells=seitz_reference_eos_n_fcc_cells,
)

selected_eos_states[[
    "n_fcc_cells",
    "target_rho",
    "actual_rho",
    "kT",
    pressure_col,
    "log_path",
]]


EOS sweep: n=10 kT=0.800 rho=0.710 direction=up
Loaded existing thermalized state:
/exp/e961/data/MDsims-data/pnichols/Thermalized_States_v3/FCC/n_cells_10/rho_0.710/kT_0.800/nsteps_1000000/seed_1/randomization.gsd
EOS sweep: n=10 kT=0.800 rho=0.715 direction=up
Created new FCC lattice
----------------------------
n_fcc_cells = 10
N = 4000
Target rho = 0.715
Actual rho = 0.7150000000000004
Density error = 4.440892098500626e-16
BoxLength = 17.75216461801366
FCC cell size = 1.775216461801366
Saved lattice to:
/exp/e961/data/MDsims-data/pnichols/Simple_Lattices_v3/FCC/n_cells_10/rho_0.715/lattice.gsd
Using GPU device
Final device: <hoomd.device.GPU object at 0x7fd054727620>
Started HDF5 logger
Log file: /exp/e961/data/MDsims-data/pnichols/Thermalized_States_v3/FCC/n_cells_10/rho_0.715/kT_0.800/nsteps_1000000/seed_1/randomization_log.hdf5
Log period: 1000
Stopped HDF5 logger
Wrote HDF5 metadata
Log file: /exp/e961/data/MDsims-data/pnichols/Thermalized_States_v3/FCC/n_cells_10/rho_0.715/kT_

In [ ]:
seitz_rows = []

for _, eos_row in selected_eos_states.sort_values([
    "n_fcc_cells",
    "target_rho",
]).iterrows():
    n_cells = int(eos_row["n_fcc_cells"])
    density = float(eos_row["target_rho"])
    nbins = seitz.nbins_for_ncells(n_cells)

    for radius in seitz_starting_radii:
        for evolve_seed in seitz_evolve_seeds:
            base_row = {
                "n_fcc_cells": n_cells,
                "N_source": 4 * n_cells ** 3,
                "source_rho": density,
                "source_kT": seitz_kT,
                "source_nsteps": seitz_source_nsteps,
                "source_seed": seitz_source_seed,
                "radius": float(radius),
                "evolve_nsteps": seitz_evolve_nsteps,
                "evolve_seed": int(evolve_seed),
                "selection_pressure": float(eos_row[pressure_col]),
                "selection_eos_n_fcc_cells": n_cells,
                "reference_eos_n_fcc_cells": int(seitz_reference_eos_n_fcc_cells),
                "voxel_nbins": int(nbins),
            }

            try:
                bubble = cavitation.get_or_create_cavitation(
                    n_fcc_cells=n_cells,
                    target_rho=density,
                    kT=seitz_kT,
                    source_nsteps=seitz_source_nsteps,
                    source_seed=seitz_source_seed,
                    radius=radius,
                    evolve_nsteps=seitz_evolve_nsteps,
                    evolve_kT=seitz_kT,
                    evolve_seed=evolve_seed,
                    log_period=seitz_log_period,
                    trajectory_period=seitz_trajectory_period,
                    reject_phase_separated_source=True,
                    classification_kwargs={"nbins": nbins},
                )

                terms = seitz.extract_bubble_state_terms(
                    bubble,
                    nbins=nbins,
                    eos_table=reference_eos_liquid,
                    n_last=seitz_n_last,
                    plot=False,
                    animate=False,
                )

                row = {
                    **base_row,
                    "status": terms.get("status", "seitz_computed"),
                    "cavitation_status": terms.get("cavitation_status", bubble.get("status")),
                    "Q": terms.get("Q"),
                    "Nc": terms.get("Nc"),
                    "uc": terms.get("uc"),
                    "u0": terms.get("u0"),
                    "P0": terms.get("P0"),
                    "rho_c": terms.get("rho_c"),
                    "rho_0": terms.get("rho_0"),
                    "log_path": str(bubble.get("paths", {}).get("log_path", "")),
                    "trajectory_path": str(bubble.get("paths", {}).get("trajectory_path", "")),
                    "reason": terms.get("reason", ""),
                }
            except Exception as error:
                row = {
                    **base_row,
                    "status": "failed",
                    "error": repr(error),
                }

            seitz_rows.append(row)
            seitz_summary_path.parent.mkdir(parents=True, exist_ok=True)
            pd.DataFrame(seitz_rows).to_csv(seitz_summary_path, index=False)

seitz_summary = pd.DataFrame(seitz_rows)
seitz_summary


In [ ]:
seitz_summary.groupby([
    "n_fcc_cells",
    "source_rho",
    "radius",
    "status",
], dropna=False).size().reset_index(name="count")